# Hybrid Search

**BM25 + Semantic Retrieval with Reciprocal Rank Fusion (RRF)**

This notebook implements a hybrid retrieval approach that combines BM25 keyword-based search with multilingual Sentence Transformer embeddings.

Reciprocal Rank Fusion (RRF) is used to combine both retrieval rankings and identify relevant historical graduation projects.

The hybrid approach will be evaluated against TF-IDF and standalone Semantic Search using the same evaluation dataset.

## 1. Setup and Dependencies

Install the libraries required for BM25, semantic embeddings, and hybrid retrieval.

In [1]:
%pip install -q sentence-transformers rank-bm25

In [2]:
import pandas as pd
import numpy as np
import re
import torch

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

print("Libraries imported successfully!")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Libraries imported successfully!
Device: cuda


## 2. Load and Validate Dataset

Load the cleaned graduation projects dataset from GitHub and verify its structure before hybrid retrieval.

In [3]:
# Load dataset
url = (
    "https://raw.githubusercontent.com/"
    "esrrahafase-crypto/Origin/main/"
    "data/processed/projects_clean.csv"
)

df = pd.read_csv(url, encoding="utf-8-sig")

# Standardize columns
df.columns = df.columns.str.strip().str.lower()
df["department"] = df["department"].str.strip()

# Validate dataset
assert df.shape == (103, 7)
assert df["project_id"].is_unique
assert df[["title", "abstract"]].notna().all().all()
assert df[["title", "abstract"]].apply(
    lambda col: col.astype(str).str.strip().ne("").all()
).all()

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Departments:", df["department"].nunique())

display(df.head(3))

Dataset loaded successfully!
Shape: (103, 7)
Departments: 6


,project_id,title,abstract,department,year,keywords,source_page
0,P001,Automatic Plant Watering and Tracking System,Many people struggle with watering their plant...,Computer Science,2024 – 2025,"Moisturem Air humidity, AWTS system, Waterin, ...",5
1,P002,علامة CourseAssist: CLO Performance Assistant ...,The rapid evolution of educational technology ...,Computer Science,2024 – 2025,"Program Learning Outcomes, Course Learning Out...",6
2,P003,واصف Mobile Application for Deaf and Mute indi...,"Our System, Wasif, offers a communication appl...",Computer Science,2024 – 2025,"Arabic Sign Language, Sign Language Gestures i...",7


## 3. Text Preparation

Prepare project titles and abstracts for semantic retrieval and BM25 keyword-based search.

In [4]:
# Prepare project texts
hybrid_df = df.copy()

hybrid_df["text"] = (
    hybrid_df["title"].str.strip()
    + ". "
    + hybrid_df["abstract"].str.strip()
)

hybrid_df["text"] = (
    hybrid_df["text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

project_texts = hybrid_df["text"].tolist()

# Tokenize for BM25
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

tokenized_corpus = [
    tokenize(text) for text in project_texts
]

print("Total project texts:", len(project_texts))
print("Total tokenized documents:", len(tokenized_corpus))
print("Sample tokens:", tokenized_corpus[0][:15])

Total project texts: 103
Total tokenized documents: 103
Sample tokens: ['automatic', 'plant', 'watering', 'and', 'tracking', 'system', 'many', 'people', 'struggle', 'with', 'watering', 'their', 'plants', 'regularly', 'which']


## 4. BM25 Retrieval Model

Build a BM25 index using the tokenized graduation project descriptions.

In [5]:
# Build BM25 index
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index created successfully!")
print("Indexed projects:", len(tokenized_corpus))

BM25 index created successfully!
Indexed projects: 103


## 5. BM25 Retrieval Function

Implement keyword-based retrieval using BM25 scores to rank the most relevant graduation projects.

In [6]:
def bm25_search(title, description, top_k=5):
    # Validate input
    if not title.strip() or not description.strip():
        raise ValueError("Title and description must not be empty.")

    # Prepare query
    query_text = f"{title.strip()}. {description.strip()}"
    query_tokens = tokenize(query_text)

    # Calculate BM25 scores
    scores = bm25.get_scores(query_tokens)

    # Retrieve Top-K
    top_indices = np.argsort(-scores)[:top_k]

    results = hybrid_df.iloc[top_indices][
        ["project_id", "title", "abstract", "department", "source_page"]
    ].copy()

    results.insert(0, "rank", range(1, len(results) + 1))
    results["bm25_score"] = scores[top_indices]

    return results.reset_index(drop=True)


print("BM25 retrieval function is ready!")

BM25 retrieval function is ready!


## 6. BM25 Retrieval Testing

Test BM25 retrieval using the same English project idea evaluated in the Semantic Search notebook.

In [7]:
# Test English query
test_title = "AI-Based Early Detection of Parkinson's Disease"

test_description = (
    "A mobile application that uses machine learning "
    "and wearable sensor data to detect early signs "
    "of Parkinson's disease and monitor patient symptoms."
)

bm25_results = bm25_search(
    title=test_title,
    description=test_description,
    top_k=5
)

display(
    bm25_results[
        ["rank", "project_id", "title", "bm25_score"]
    ].round(4)
)

,rank,project_id,title,bm25_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,65.4469
1,2,P005,Thabat – ثبات,45.7186
2,3,P080,A Smart Mobile Application For Skin Cancer Det...,26.5601
3,4,P070,Mama’s Hope,25.5315
4,5,P026,Potato Guard,24.6065


## 7. Load Semantic Retrieval Model

Load the same pretrained multilingual Sentence Transformer used in the standalone Semantic Search implementation.

In [8]:
# Load semantic model
model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(model_name, device=device)

print("Semantic model loaded successfully!")
print("Device:", device)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic model loaded successfully!
Device: cuda
Embedding dimension: 768


/tmp/ipykernel_1605/3673033430.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


## 8. Generate Semantic Embeddings

Generate normalized semantic embeddings for all graduation projects to enable similarity-based retrieval.

In [9]:
# Generate project embeddings
project_embeddings = model.encode(
    project_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Validate embeddings
assert project_embeddings.shape == (
    len(project_texts),
    model.get_sentence_embedding_dimension()
)

assert np.isfinite(project_embeddings).all()

print("Semantic embeddings generated successfully!")
print("Embeddings shape:", project_embeddings.shape)
print("Data type:", project_embeddings.dtype)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Semantic embeddings generated successfully!
Embeddings shape: (103, 768)
Data type: float32


/tmp/ipykernel_1605/710503941.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


## 9. Semantic Retrieval Function

Implement semantic retrieval using normalized Sentence Transformer embeddings and Cosine Similarity.

In [10]:
def semantic_search(title, description, top_k=5):
    # Validate input
    if not title.strip() or not description.strip():
        raise ValueError("Title and description must not be empty.")

    # Prepare query
    query_text = f"{title.strip()}. {description.strip()}"

    # Encode query
    query_embedding = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Calculate similarity
    scores = cosine_similarity(
        query_embedding,
        project_embeddings
    )[0]

    # Retrieve Top-K
    top_indices = np.argsort(-scores)[:top_k]

    results = hybrid_df.iloc[top_indices][
        ["project_id", "title", "abstract", "department", "source_page"]
    ].copy()

    results.insert(0, "rank", range(1, len(results) + 1))
    results["similarity_score"] = scores[top_indices]

    return results.reset_index(drop=True)


print("Semantic retrieval function is ready!")

Semantic retrieval function is ready!


## 10. Reciprocal Rank Fusion (RRF)

Combine BM25 and semantic retrieval rankings using Reciprocal Rank Fusion to produce a unified hybrid ranking.

In [17]:
def hybrid_search(title, description, top_k=5, rrf_k=60):
    # Validate input
    if not title.strip() or not description.strip():
        raise ValueError("Title and description must not be empty.")

    if not 1 <= top_k <= len(hybrid_df):
        raise ValueError("top_k must be between 1 and the dataset size.")

    if rrf_k <= 0:
        raise ValueError("rrf_k must be positive.")

    # Get semantic rankings
    semantic_results = semantic_search(
        title, description, top_k=len(hybrid_df)
    )

    semantic_ranks = semantic_results.set_index("project_id")["rank"]

    # Calculate BM25 scores
    query_text = f"{title.strip()}. {description.strip()}"
    bm25_scores = bm25.get_scores(tokenize(query_text))

    results = hybrid_df[
        ["project_id", "title", "abstract", "department", "source_page"]
    ].copy()

    results["semantic_rank"] = results["project_id"].map(semantic_ranks)

    # Use semantic retrieval when BM25 has no matching terms
    if not np.any(bm25_scores > 0):
        results["bm25_rank"] = np.nan
        results["rrf_score"] = 1 / (rrf_k + results["semantic_rank"])
        print("BM25 has no positive matches; using semantic retrieval only.")

    else:
        bm25_results = bm25_search(
            title, description, top_k=len(hybrid_df)
        )

        bm25_ranks = bm25_results.set_index("project_id")["rank"]

        results["bm25_rank"] = results["project_id"].map(bm25_ranks)

        results["rrf_score"] = (
            1 / (rrf_k + results["bm25_rank"])
            + 1 / (rrf_k + results["semantic_rank"])
        )

    # Retrieve Top-K
    results = results.sort_values(
        ["rrf_score", "project_id"],
        ascending=[False, True]
    ).head(top_k).copy()

    results.insert(0, "rank", range(1, len(results) + 1))

    return results.reset_index(drop=True)


print("Hybrid retrieval function updated successfully!")

Hybrid retrieval function updated successfully!


## 11. Hybrid Search Testing

Evaluate the hybrid retrieval pipeline using the same English project idea tested with BM25 and Semantic Search.

In [12]:
# Test hybrid retrieval
hybrid_results = hybrid_search(
    title=test_title,
    description=test_description,
    top_k=5
)

display(
    hybrid_results[
        [
            "rank",
            "project_id",
            "title",
            "bm25_rank",
            "semantic_rank",
            "rrf_score"
        ]
    ].round(4)
)

,rank,project_id,title,bm25_rank,semantic_rank,rrf_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,1,1,0.0328
1,2,P080,A Smart Mobile Application For Skin Cancer Det...,3,2,0.0320
2,3,P005,Thabat – ثبات,2,4,0.0318
3,4,P082,Predicting Diabetes Patients Based On Machine ...,6,5,0.0305
4,5,P089,AI-Powered Platform for Plant Identification a...,18,3,0.0287


## 12. Arabic Hybrid Search Testing

Test the hybrid retrieval pipeline using an Arabic project idea to examine cross-lingual retrieval behavior.

In [13]:
# Test Arabic query
arabic_title = "الكشف المبكر عن مرض باركنسون باستخدام الذكاء الاصطناعي"

arabic_description = (
    "تطبيق يستخدم تعلم الآلة وبيانات أجهزة الاستشعار "
    "القابلة للارتداء للكشف المبكر عن مرض باركنسون "
    "ومراقبة أعراض المرض لدى المرضى."
)

arabic_hybrid_results = hybrid_search(
    title=arabic_title,
    description=arabic_description,
    top_k=5
)

display(
    arabic_hybrid_results[
        [
            "rank",
            "project_id",
            "title",
            "bm25_rank",
            "semantic_rank",
            "rrf_score"
        ]
    ].round(4)
)

,rank,project_id,title,bm25_rank,semantic_rank,rrf_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,25,1,0.0282
1,2,P010,Dexter,10,13,0.0280
2,3,P005,Thabat – ثبات,5,30,0.0265
3,4,P016,PulseXPert-Health Disorder Detection,16,15,0.0265
4,5,P024,Crime analysis and prediction,24,10,0.0262


## 13. Arabic BM25 Diagnostic Analysis

Examine BM25 score coverage for Arabic queries to identify potential cross-lingual retrieval limitations.

In [14]:
# Analyze Arabic BM25 scores
arabic_query = f"{arabic_title}. {arabic_description}"
arabic_tokens = tokenize(arabic_query)

arabic_bm25_scores = bm25.get_scores(arabic_tokens)

print("Total projects:", len(arabic_bm25_scores))
print("Projects with positive scores:", np.sum(arabic_bm25_scores > 0))
print("Projects with zero scores:", np.sum(arabic_bm25_scores == 0))

# Inspect hybrid result scores
diagnostic = arabic_hybrid_results[
    ["rank", "project_id", "title", "bm25_rank", "semantic_rank"]
].copy()

diagnostic["bm25_score"] = diagnostic["project_id"].map(
    dict(zip(hybrid_df["project_id"], arabic_bm25_scores))
)

display(diagnostic.round(4))

Total projects: 103
Projects with positive scores: 0
Projects with zero scores: 103


,rank,project_id,title,bm25_rank,semantic_rank,bm25_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,25,1,0.0
1,2,P010,Dexter,10,13,0.0
2,3,P005,Thabat – ثبات,5,30,0.0
3,4,P016,PulseXPert-Health Disorder Detection,16,15,0.0
4,5,P024,Crime analysis and prediction,24,10,0.0


In [18]:
# Re-test Arabic hybrid retrieval
arabic_hybrid_results_fixed = hybrid_search(
    title=arabic_title,
    description=arabic_description,
    top_k=5
)

display(
    arabic_hybrid_results_fixed[
        [
            "rank",
            "project_id",
            "title",
            "bm25_rank",
            "semantic_rank",
            "rrf_score"
        ]
    ].round(4)
)

BM25 has no positive matches; using semantic retrieval only.


,rank,project_id,title,bm25_rank,semantic_rank,rrf_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,NaN,1,0.0164
1,2,P082,Predicting Diabetes Patients Based On Machine ...,NaN,2,0.0161
2,3,P059,Blood Bank Inventory System - Artificial Intel...,NaN,3,0.0159
3,4,P089,AI-Powered Platform for Plant Identification a...,NaN,4,0.0156
4,5,P080,A Smart Mobile Application For Skin Cancer Det...,NaN,5,0.0154


## 14. Hybrid Retrieval Validation

Validate hybrid retrieval rankings, result uniqueness, and semantic fallback behavior for Arabic queries.

In [19]:
# Re-test English hybrid retrieval
english_hybrid_results = hybrid_search(
    test_title, test_description, top_k=5
)

# Validate retrieval outputs
for results in [english_hybrid_results, arabic_hybrid_results_fixed]:
    assert len(results) == 5
    assert results["project_id"].is_unique
    assert results["rank"].tolist() == [1, 2, 3, 4, 5]
    assert results["rrf_score"].is_monotonic_decreasing
    assert np.isfinite(results["rrf_score"]).all()

# Validate Arabic semantic fallback
arabic_semantic_results = semantic_search(
    arabic_title, arabic_description, top_k=5
)

assert arabic_hybrid_results_fixed["bm25_rank"].isna().all()

assert (
    arabic_hybrid_results_fixed["project_id"].tolist()
    == arabic_semantic_results["project_id"].tolist()
)

print("All hybrid retrieval validation checks passed!")

All hybrid retrieval validation checks passed!


## 15. Save Hybrid Retrieval Results

Export English and Arabic hybrid retrieval results for reproducibility and future evaluation.

In [20]:
from pathlib import Path

# Create output directory
output_dir = Path("/content/origin_hybrid_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Save retrieval results
english_hybrid_results.to_csv(
    output_dir / "hybrid_english_results.csv",
    index=False,
    encoding="utf-8-sig"
)

arabic_hybrid_results_fixed.to_csv(
    output_dir / "hybrid_arabic_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Hybrid retrieval results saved successfully!")

for file in sorted(output_dir.iterdir()):
    print(file.name)

Hybrid retrieval results saved successfully!
hybrid_arabic_results.csv
hybrid_english_results.csv


In [21]:
import shutil
from google.colab import files

# Create ZIP archive
archive_path = shutil.make_archive(
    "/content/origin_hybrid_outputs",
    "zip",
    output_dir
)

files.download(archive_path)

print("Hybrid search outputs downloaded successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Hybrid search outputs downloaded successfully!


## 16. Conclusions

This notebook implemented a hybrid retrieval system combining BM25 keyword-based search and multilingual Sentence Transformer embeddings using Reciprocal Rank Fusion (RRF).

### Key Findings

- The hybrid system successfully retrieved and ranked graduation projects using English and Arabic queries.
- For the English Parkinson's disease query, project P025 ranked first in both retrieval methods and the hybrid results.
- For the Arabic query, all 103 projects received zero BM25 scores, revealing a limitation in cross-lingual keyword matching.
- A semantic-only fallback was implemented to prevent uninformative BM25 rankings from affecting the final results.
- Retrieval validation checks passed for both English and Arabic queries.

### Limitations and Future Improvements

- BM25 currently uses basic tokenization without advanced linguistic preprocessing.
- The current fallback handles queries with no positive BM25 matches; partial zero-score matches still require further investigation.
- Multilingual keyword retrieval could be improved through query translation or other cross-lingual techniques.
- Final retrieval quality must be evaluated using human-annotated queries and metrics such as Precision@K, Recall@K, and MRR.

These results establish a functional hybrid retrieval baseline for the Origin system. Further evaluation and optimization are required before determining whether hybrid retrieval outperforms standalone approaches.